# TRAINING MODEL MAKING PREDICTIONS

In [1]:
import sys
from pathlib import Path

In [2]:
import pandas as pd

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [4]:
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from xgboost import XGBRegressor

In [5]:
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
import matplotlib.pyplot as plt

In [6]:
project_root = Path.cwd().parent

In [7]:
sys.path.append(str(project_root))

In [8]:
data_path = project_root / "data" / "cleaned_datasets"

In [9]:
energy_df = pd.read_csv(data_path / "energy_clean.csv")

In [10]:
energy_df.head()

,DateTime,generation_biomass,generation_fossil_brown_coal/lignite,generation_fossil_gas,generation_fossil_hard_coal,generation_fossil_oil,generation_hydro_pumped_storage_consumption,generation_hydro_run-of-river_and_poundage,generation_hydro_water_reservoir,generation_nuclear,...,total_load_actual,price_day_ahead,price_actual,total_generation,generation_load_difference,total_load_rolling,total_gen_rolling,day_of_week,hour_of_day,month
0,2014-12-31 23:00:00,447.0,329.0,4844.0,4821.0,162.0,863.0,1051.0,1899.0,7096.0,...,25385.0,50.10,65.41,28251.0,2866.0,NaN,NaN,2,23,12
1,2015-01-01 00:00:00,449.0,328.0,5196.0,4755.0,158.0,920.0,1009.0,1658.0,7096.0,...,24382.0,48.10,64.92,27818.0,3436.0,NaN,NaN,3,0,1
2,2015-01-01 01:00:00,448.0,323.0,4857.0,4581.0,157.0,1164.0,973.0,1371.0,7099.0,...,22734.0,47.33,64.48,26796.0,4062.0,NaN,NaN,3,1,1
3,2015-01-01 02:00:00,438.0,254.0,4314.0,4131.0,160.0,1503.0,949.0,779.0,7098.0,...,21286.0,42.27,59.32,25223.0,3937.0,NaN,NaN,3,2,1
4,2015-01-01 03:00:00,428.0,187.0,4130.0,3840.0,156.0,1826.0,953.0,720.0,7097.0,...,20264.0,38.41,56.04,24620.0,4356.0,NaN,NaN,3,3,1


In [11]:
if energy_df["DateTime"].nunique() == len(energy_df):
    energy_df = energy_df.set_index("DateTime")
    print("DateTime set as index successfully!")
else:
    print("Warning: DateTime column has duplicates. Consider resolving them first.")

DateTime set as index successfully!


In [12]:
energy_df.sort_index(inplace=True)

In [13]:
energy_df.head()

,generation_biomass,generation_fossil_brown_coal/lignite,generation_fossil_gas,generation_fossil_hard_coal,generation_fossil_oil,generation_hydro_pumped_storage_consumption,generation_hydro_run-of-river_and_poundage,generation_hydro_water_reservoir,generation_nuclear,generation_other,...,total_load_actual,price_day_ahead,price_actual,total_generation,generation_load_difference,total_load_rolling,total_gen_rolling,day_of_week,hour_of_day,month
DateTime,,,,,,,,,,,,,,,,,,,,,
2014-12-31 23:00:00,447.0,329.0,4844.0,4821.0,162.0,863.0,1051.0,1899.0,7096.0,43.0,...,25385.0,50.10,65.41,28251.0,2866.0,NaN,NaN,2,23,12
2015-01-01 00:00:00,449.0,328.0,5196.0,4755.0,158.0,920.0,1009.0,1658.0,7096.0,43.0,...,24382.0,48.10,64.92,27818.0,3436.0,NaN,NaN,3,0,1
2015-01-01 01:00:00,448.0,323.0,4857.0,4581.0,157.0,1164.0,973.0,1371.0,7099.0,43.0,...,22734.0,47.33,64.48,26796.0,4062.0,NaN,NaN,3,1,1
2015-01-01 02:00:00,438.0,254.0,4314.0,4131.0,160.0,1503.0,949.0,779.0,7098.0,43.0,...,21286.0,42.27,59.32,25223.0,3937.0,NaN,NaN,3,2,1
2015-01-01 03:00:00,428.0,187.0,4130.0,3840.0,156.0,1826.0,953.0,720.0,7097.0,43.0,...,20264.0,38.41,56.04,24620.0,4356.0,NaN,NaN,3,3,1


In [14]:
energy_df.columns

Index(['generation_biomass', 'generation_fossil_brown_coal/lignite',
       'generation_fossil_gas', 'generation_fossil_hard_coal',
       'generation_fossil_oil', 'generation_hydro_pumped_storage_consumption',
       'generation_hydro_run-of-river_and_poundage',
       'generation_hydro_water_reservoir', 'generation_nuclear',
       'generation_other', 'generation_other_renewable', 'generation_solar',
       'generation_waste', 'generation_wind_onshore',
       'forecast_solar_day_ahead', 'forecast_wind_onshore_day_ahead',
       'total_load_forecast', 'total_load_actual', 'price_day_ahead',
       'price_actual', 'total_generation', 'generation_load_difference',
       'total_load_rolling', 'total_gen_rolling', 'day_of_week', 'hour_of_day',
       'month'],
      dtype='object')

In [15]:
training_columns = ['price_actual', 'price_day_ahead', "generation_fossil_brown_coal/lignite",
                    "generation_fossil_gas","generation_fossil_hard_coal","generation_solar",
                    "generation_wind_onshore","total_load_actual","generation_nuclear","generation_hydro_water_reservoir"]

In [16]:
training_columns_with_date = ['price_actual', 'price_day_ahead', "generation_fossil_brown_coal/lignite",
                    "generation_fossil_gas","generation_fossil_hard_coal","generation_solar",
                    "generation_wind_onshore","total_load_actual","generation_nuclear","generation_hydro_water_reservoir",
                    'day_of_week', 'hour_of_day', 'date']

In [17]:
energy_selected = energy_df[training_columns]

In [18]:
energy_selected.head()

,price_actual,price_day_ahead,generation_fossil_brown_coal/lignite,generation_fossil_gas,generation_fossil_hard_coal,generation_solar,generation_wind_onshore,total_load_actual,generation_nuclear,generation_hydro_water_reservoir
DateTime,,,,,,,,,,
2014-12-31 23:00:00,65.41,50.10,329.0,4844.0,4821.0,49.0,6378.0,25385.0,7096.0,1899.0
2015-01-01 00:00:00,64.92,48.10,328.0,5196.0,4755.0,50.0,5890.0,24382.0,7096.0,1658.0
2015-01-01 01:00:00,64.48,47.33,323.0,4857.0,4581.0,50.0,5461.0,22734.0,7099.0,1371.0
2015-01-01 02:00:00,59.32,42.27,254.0,4314.0,4131.0,50.0,5238.0,21286.0,7098.0,779.0
2015-01-01 03:00:00,56.04,38.41,187.0,4130.0,3840.0,42.0,4935.0,20264.0,7097.0,720.0


In [19]:
energy_selected['price_prev'] = energy_selected['price_actual'].shift(24)

C:\Users\Milos\AppData\Local\Temp\ipykernel_10264\4143429232.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  energy_selected['price_prev'] = energy_selected['price_actual'].shift(24)


In [20]:
energy_selected.head()

,price_actual,price_day_ahead,generation_fossil_brown_coal/lignite,generation_fossil_gas,generation_fossil_hard_coal,generation_solar,generation_wind_onshore,total_load_actual,generation_nuclear,generation_hydro_water_reservoir,price_prev
DateTime,,,,,,,,,,,
2014-12-31 23:00:00,65.41,50.10,329.0,4844.0,4821.0,49.0,6378.0,25385.0,7096.0,1899.0,NaN
2015-01-01 00:00:00,64.92,48.10,328.0,5196.0,4755.0,50.0,5890.0,24382.0,7096.0,1658.0,NaN
2015-01-01 01:00:00,64.48,47.33,323.0,4857.0,4581.0,50.0,5461.0,22734.0,7099.0,1371.0,NaN
2015-01-01 02:00:00,59.32,42.27,254.0,4314.0,4131.0,50.0,5238.0,21286.0,7098.0,779.0,NaN
2015-01-01 03:00:00,56.04,38.41,187.0,4130.0,3840.0,42.0,4935.0,20264.0,7097.0,720.0,NaN


In [21]:
energy_selected.dropna(inplace=True)

C:\Users\Milos\AppData\Local\Temp\ipykernel_10264\4000456823.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  energy_selected.dropna(inplace=True)


In [22]:
energy_selected.isna().sum()

price_actual                            0
price_day_ahead                         0
generation_fossil_brown_coal/lignite    0
generation_fossil_gas                   0
generation_fossil_hard_coal             0
generation_solar                        0
generation_wind_onshore                 0
total_load_actual                       0
generation_nuclear                      0
generation_hydro_water_reservoir        0
price_prev                              0
dtype: int64

In [23]:
# Create day-ahead price features
# Generate 24 hourly features representing future prices starting 10 periods ahead (midnight)
# Set up iterators for hours and corresponding shifts
hours = list(range(24))                     # Hour indices: 0 to 23
day_ahead_shifts = list(range(0, -24, -1))  # Shifts for aligning future prices

# Loop through shifts to create new columns for each hour
for shift_value, hour in zip(day_ahead_shifts, hours):
    column_name = f't_price_{hour}'
    energy_selected[column_name] = energy_selected['price_actual'].shift(shift_value)

C:\Users\Milos\AppData\Local\Temp\ipykernel_10264\3873220131.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  energy_selected[column_name] = energy_selected['price_actual'].shift(shift_value)
C:\Users\Milos\AppData\Local\Temp\ipykernel_10264\3873220131.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  energy_selected[column_name] = energy_selected['price_actual'].shift(shift_value)
C:\Users\Milos\AppData\Local\Temp\ipykernel_10264\3873220131.py:10: SettingWithCopyWarning: 
A value is trying to be set

In [24]:
energy_selected.head()

,price_actual,price_day_ahead,generation_fossil_brown_coal/lignite,generation_fossil_gas,generation_fossil_hard_coal,generation_solar,generation_wind_onshore,total_load_actual,generation_nuclear,generation_hydro_water_reservoir,...,t_price_14,t_price_15,t_price_16,t_price_17,t_price_18,t_price_19,t_price_20,t_price_21,t_price_22,t_price_23
DateTime,,,,,,,,,,,,,,,,,,,,,
2015-01-01 23:00:00,64.02,10.0,0.0,3189.0,1291.0,54.0,13232.0,27070.0,7106.0,3871.0,...,71.04,71.24,70.64,72.85,82.55,83.33,83.23,79.06,76.20,71.75
2015-01-02 00:00:00,58.46,7.0,0.0,2902.0,1190.0,54.0,12760.0,24935.0,7106.0,2996.0,...,71.24,70.64,72.85,82.55,83.33,83.23,79.06,76.20,71.75,66.82
2015-01-02 01:00:00,54.70,5.0,0.0,2772.0,1023.0,54.0,12174.0,23214.0,7107.0,2581.0,...,70.64,72.85,82.55,83.33,83.23,79.06,76.20,71.75,66.82,63.35
2015-01-02 02:00:00,54.91,4.0,0.0,2936.0,1016.0,53.0,11358.0,22540.0,7107.0,2933.0,...,72.85,82.55,83.33,83.23,79.06,76.20,71.75,66.82,63.35,58.79
2015-01-02 03:00:00,53.07,4.0,0.0,2893.0,1103.0,53.0,11199.0,22096.0,7106.0,2646.0,...,82.55,83.33,83.23,79.06,76.20,71.75,66.82,63.35,58.79,57.44


In [25]:
energy_selected.drop(columns = ["price_actual","price_day_ahead"],inplace = True)

C:\Users\Milos\AppData\Local\Temp\ipykernel_10264\2437129929.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  energy_selected.drop(columns = ["price_actual","price_day_ahead"],inplace = True)


In [26]:
energy_selected.columns

Index(['generation_fossil_brown_coal/lignite', 'generation_fossil_gas',
       'generation_fossil_hard_coal', 'generation_solar',
       'generation_wind_onshore', 'total_load_actual', 'generation_nuclear',
       'generation_hydro_water_reservoir', 'price_prev', 't_price_0',
       't_price_1', 't_price_2', 't_price_3', 't_price_4', 't_price_5',
       't_price_6', 't_price_7', 't_price_8', 't_price_9', 't_price_10',
       't_price_11', 't_price_12', 't_price_13', 't_price_14', 't_price_15',
       't_price_16', 't_price_17', 't_price_18', 't_price_19', 't_price_20',
       't_price_21', 't_price_22', 't_price_23'],
      dtype='object')

In [27]:
y_cols = [col for col in energy_selected.columns if col.startswith('t_price')]

In [28]:
# Set X and y
X = energy_selected.drop(columns=y_cols)
y = energy_selected[y_cols]

In [29]:
# Define the proportion of the dataset to use as the test set
test_size = 0.2  # 20% of the data will be used for testing/validation

# Split the dataset into training and test sets
# shuffle=False is important for time series to preserve temporal order
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=test_size, 
    shuffle=False
)

In [30]:
# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Fit the scaler on the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test data using the same scaler (do not fit again)
X_test_scaled = scaler.transform(X_test)

In [31]:
# Define cross-validation parameters
num_folds = 5
error_metrics = ['neg_root_mean_squared_error', 'r2']  # Metrics for evaluation

# Define a dictionary of models
models = {
    'MLP': MLPRegressor(),
    'RFR': RandomForestRegressor(),
    'SVR': SVR(),
    'AdaB': AdaBoostRegressor(),
    'GBR': GradientBoostingRegressor(),
    'DTR': DecisionTreeRegressor(),
    'XGB': XGBRegressor(),
    'LR': LinearRegression(),
    'EN': ElasticNet()
}

In [36]:
from sklearn.impute import SimpleImputer

imputer_X = SimpleImputer(strategy='mean')
X_train_scaled = imputer_X.fit_transform(X_train_scaled)
X_test_scaled = imputer_X.transform(X_test_scaled)

imputer_y = SimpleImputer(strategy='mean')
y_train = imputer_y.fit_transform(y_train)
y_test = imputer_y.transform(y_test)

In [37]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Initialize Linear Regression
linear_regression = LinearRegression(fit_intercept=False, positive=False)

# Fit the model to the training data
linear_regression.fit(X_train_scaled, y_train)

# Make predictions on the test data
y_pred_lr = linear_regression.predict(X_test_scaled)

# Overall evaluation
rmse_overall = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_overall = r2_score(y_test, y_pred_lr)

print(f"Linear Regression Multi-Output Results (Overall):")
print(f"RMSE: {rmse_overall:.4f}")
print(f"R²: {r2_overall:.4f}\n")

# Per-hour evaluation
hourly_metrics = []
for hour in range(y_test.shape[1]):
    rmse_hour = np.sqrt(mean_squared_error(y_test[:, hour], y_pred_lr[:, hour]))
    r2_hour = r2_score(y_test[:, hour], y_pred_lr[:, hour])
    hourly_metrics.append((hour, rmse_hour, r2_hour))

# Convert to DataFrame for easy inspection
hourly_metrics_df = pd.DataFrame(hourly_metrics, columns=['Hour', 'RMSE', 'R2'])
print("Per-hour performance metrics:")
print(hourly_metrics_df)

# Summary statistics
print("\nSummary across 24 hours:")
print(hourly_metrics_df[['RMSE', 'R2']].describe())


Linear Regression Multi-Output Results (Overall):
RMSE: 12.2018
R²: -0.1291

Per-hour performance metrics:
    Hour       RMSE        R2
0      0   7.356064  0.590556
1      1   7.955279  0.520965
2      2   9.223890  0.355960
3      3  10.605258  0.148580
4      4  11.821556 -0.057956
5      5  12.744714 -0.229704
6      6  13.340538 -0.347500
7      7  13.605793 -0.401885
8      8  13.558452 -0.392685
9      9  13.251896 -0.331067
10    10  12.762646 -0.235215
11    11  12.259690 -0.140140
12    12  11.976546 -0.088059
13    13  12.066602 -0.104388
14    14  12.527647 -0.190289
15    15  13.151319 -0.311722
16    16  13.696539 -0.423003
17    17  14.016525 -0.490946
18    18  14.013426 -0.490381
19    19  13.665515 -0.417763
20    20  12.984551 -0.280634
21    21  12.029792 -0.099936
22    22  10.961634  0.086106
23    23  10.038217  0.233285

Summary across 24 hours:
            RMSE         R2
count  24.000000  24.000000
mean   12.067254  -0.129076
std     1.845735   0.308138
min  

In [38]:
import plotly.graph_objects as go

# Choose which hour to plot (0-23)
hour_to_plot = 0

# Extract series for that hour
y_test_hour = y_test[:, hour_to_plot]
y_pred_hour = y_pred_lr[:, hour_to_plot]

# If you have a datetime index for test set
time_index = X_test.index if hasattr(X_test, 'index') else range(len(y_test_hour))

# Create the figure
fig = go.Figure()

# True values
fig.add_trace(go.Scatter(x=time_index, y=y_test_hour, mode='lines', name='True Values',
                         line=dict(color='cyan', width=2)))

# Predicted values
fig.add_trace(go.Scatter(x=time_index, y=y_pred_hour, mode='lines', name='Predicted Values',
                         line=dict(color='orange', width=2, dash='dot')))

# Layout
fig.update_layout(
    title=f'True vs Predicted Electricity Price (Hour {hour_to_plot})',
    xaxis_title='Date/Time',
    yaxis_title='Price (Euro/kWh)',
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(
        x=0.01, y=0.99,
        traceorder='normal',
        font=dict(family="Arial", size=12, color="white"),
        bgcolor='rgba(50, 50, 50, 0.5)',
        bordercolor='white',
        borderwidth=1
    ),
)

fig.show()


In [39]:
import numpy as np
import pandas as pd

# Example: new input values (replace with real values)
new_data = {
    'generation_fossil_brown_coal/lignite': [1000],
    'generation_fossil_gas': [500],
    'generation_fossil_hard_coal': [800],
    'generation_solar': [200],
    'generation_wind_onshore': [300],
    'total_load_actual': [4000],
    'generation_nuclear': [1000],
    'generation_hydro_water_reservoir': [150],
    'price_prev': [50]
}

# Create a DataFrame (columns must match the training features)
new_df = pd.DataFrame(new_data)

# Scale new input using the same scaler used for training
new_df_scaled = scaler.transform(new_df)  # Assuming you have `scaler = MinMaxScaler()` from before

# Predict using the trained model
y_pred_new = linear_regression.predict(new_df_scaled)  # or rf_model.predict(new_df_scaled)

print("Predicted electricity prices for 24 hours:")
print(y_pred_new)


Predicted electricity prices for 24 hours:
[[32.06785349 33.84254044 37.39143969 41.75717963 46.20851854 50.03280072
  52.66552    53.68008998 52.94529272 50.98703674 48.67057916 47.12852088
  47.11837748 48.54905333 50.87993091 53.44453707 55.34908667 55.85219477
  54.61072866 51.54413008 47.01259271 42.00528365 37.76730579 35.23959657]]
